# Reto 3 - Modelo de Grafo con Neo4j

---

## Objetivos

1. **Diseñar** un modelo de grafo con nodos, etiquetas, relaciones y atributos.
2. **Crear** el grafo en Neo4j usando Cypher con un subconjunto representativo de los datos.
3. **Ejecutar** las consultas propuestas y demostrar la utilidad del modelo de grafo.

> **Pre-requisito:** Neo4j corriendo en `localhost:7687`.   

---

## 1. Diseño del modelo de grafo

### Nodos y atributos

| Etiqueta | Atributos principales |
|----------|-----------------------|
| `Barrio` | `nombre` |
| `Local` | `nombre`, `tipo_actividad`, `horario`, `licencia` |
| `Terraza` | `nombre`, `capacidad`, `estado_licencia` |
| `Alojamiento` | `nombre`, `precio`, `numero_habitaciones`, `reseñas`, `servicios[]` |

### Relaciones y atributos

| Relación | Conecta | Atributos |
|----------|---------|----------|
| `UBICADO_EN` | Local/Terraza/Alojamiento → Barrio | `distancia` (km al centro del barrio) |
| `CERCANO_A` | Local/Terraza/Alojamiento ↔ Local/Terraza/Alojamiento | `distancia` (km entre entidades) |
| `RELACIONADO_CON` | Local/Terraza ↔ Local/Terraza/Alojamiento | `motivo` |

## 2. Importación de librerías

In [37]:
from neo4j import GraphDatabase
import neo4j

print(f"Neo4j driver version: {neo4j.__version__}")

Neo4j driver version: 6.1.0


## 3. Conexión a Neo4j

In [38]:
NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "abc123456"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Conexión a Neo4j establecida correctamente.")


def run_query(cypher, params=None, verbose=True):
    with driver.session() as session:
        results = session.run(cypher, params or {})
        data = results.data()
        if verbose:
            for row in data:
                print(row)
        return data

Conexión a Neo4j establecida correctamente.


## 4. Limpieza del grafo (opcional - reinicio limpio)

In [40]:
# ADVERTENCIA: elimina todos los nodos y relaciones del grafo
run_query("MATCH (n) DETACH DELETE n", verbose=False)
print("Grafo limpiado. Listo para crear nodos y relaciones.")

Grafo limpiado. Listo para crear nodos y relaciones.


## 5. Creación de nodos

### 5.1 Nodos de tipo `Barrio`

In [41]:
cypher_barrios = """
CREATE (b1:Barrio {nombre: "Salamanca"})
CREATE (b2:Barrio {nombre: "Chamberí"})
CREATE (b3:Barrio {nombre: "Centro"})
CREATE (b4:Barrio {nombre: "Retiro"})
RETURN 'Barrios creados' AS resultado
"""
run_query(cypher_barrios)

{'resultado': 'Barrios creados'}


[{'resultado': 'Barrios creados'}]

### 5.2 Nodos de tipo `Local`

In [42]:
cypher_locales = """
CREATE (l1:Local {
    nombre:          "Cafetería Goya",
    tipo_actividad:  "Restauración",
    horario:         "08:00-22:00",
    licencia:        "Concedida"
})
CREATE (l2:Local {
    nombre:          "Librería Central",
    tipo_actividad:  "Cultura",
    horario:         "10:00-20:00",
    licencia:        "En trámite"
})
CREATE (l3:Local {
    nombre:          "Restaurante El Patio",
    tipo_actividad:  "Restauración",
    horario:         "13:00-01:00",
    licencia:        "Concedida"
})
CREATE (l4:Local {
    nombre:          "Farmacia Mayor",
    tipo_actividad:  "Salud",
    horario:         "09:00-21:00",
    licencia:        "Concedida"
})
RETURN 'Locales creados' AS resultado
"""
run_query(cypher_locales)

{'resultado': 'Locales creados'}


[{'resultado': 'Locales creados'}]

### 5.3 Nodos de tipo `Terraza`

In [43]:
cypher_terrazas = """
CREATE (t1:Terraza {
    nombre:          "Terraza Sol",
    capacidad:       30,
    estado_licencia: "Concedida"
})
CREATE (t2:Terraza {
    nombre:          "Terraza Norte",
    capacidad:       20,
    estado_licencia: "En trámite"
})
CREATE (t3:Terraza {
    nombre:          "Terraza Retiro",
    capacidad:       50,
    estado_licencia: "Concedida"
})
RETURN 'Terrazas creadas' AS resultado
"""
run_query(cypher_terrazas)

{'resultado': 'Terrazas creadas'}


[{'resultado': 'Terrazas creadas'}]

### 5.4 Nodos de tipo `Alojamiento`

In [44]:
cypher_alojamientos = """
CREATE (a1:Alojamiento {
    nombre:               "Airbnb Retiro",
    precio:               120,
    numero_habitaciones:  2,
    reseñas:              45,
    servicios:            ["WiFi", "Cocina", "Aire acondicionado"]
})
CREATE (a2:Alojamiento {
    nombre:               "Apartamento Goya",
    precio:               80,
    numero_habitaciones:  1,
    reseñas:              0,
    servicios:            ["Aire acondicionado"]
})
CREATE (a3:Alojamiento {
    nombre:               "Suite Salamanca",
    precio:               200,
    numero_habitaciones:  3,
    reseñas:              120,
    servicios:            ["WiFi", "Piscina", "Parking", "Cocina"]
})
CREATE (a4:Alojamiento {
    nombre:               "Estudio Chamberí",
    precio:               65,
    numero_habitaciones:  0,
    reseñas:              0,
    servicios:            ["WiFi"]
})
CREATE (a5:Alojamiento {
    nombre:               "Loft Centro",
    precio:               150,
    numero_habitaciones:  0,
    reseñas:              30,
    servicios:            ["WiFi", "Cocina", "Terraza"]
})
RETURN 'Alojamientos creados' AS resultado
"""
run_query(cypher_alojamientos)

{'resultado': 'Alojamientos creados'}


[{'resultado': 'Alojamientos creados'}]

## 6. Creación de relaciones

### 6.1 Relación `UBICADO_EN` (Local/Terraza/Alojamiento → Barrio)

In [45]:
cypher_ubicado_en = [
"""
MATCH (b:Barrio {nombre: "Salamanca"}), (l:Local {nombre: "Cafetería Goya"})
CREATE (l)-[:UBICADO_EN {distancia: 0.5}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Chamberí"}), (l:Local {nombre: "Librería Central"})
CREATE (l)-[:UBICADO_EN {distancia: 0.3}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Centro"}), (l:Local {nombre: "Restaurante El Patio"})
CREATE (l)-[:UBICADO_EN {distancia: 0.1}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Salamanca"}), (l:Local {nombre: "Farmacia Mayor"})
CREATE (l)-[:UBICADO_EN {distancia: 0.8}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Salamanca"}), (t:Terraza {nombre: "Terraza Sol"})
CREATE (t)-[:UBICADO_EN {distancia: 0.4}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Centro"}), (t:Terraza {nombre: "Terraza Norte"})
CREATE (t)-[:UBICADO_EN {distancia: 0.2}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Retiro"}), (t:Terraza {nombre: "Terraza Retiro"})
CREATE (t)-[:UBICADO_EN {distancia: 0.6}]->(b)
""",
""" 
MATCH (b:Barrio {nombre: "Retiro"}), (a:Alojamiento {nombre: "Airbnb Retiro"})
CREATE (a)-[:UBICADO_EN {distancia: 0.3}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Salamanca"}), (a:Alojamiento {nombre: "Apartamento Goya"})
CREATE (a)-[:UBICADO_EN {distancia: 0.7}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Salamanca"}), (a:Alojamiento {nombre: "Suite Salamanca"})
CREATE (a)-[:UBICADO_EN {distancia: 0.2}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Chamberí"}), (a:Alojamiento {nombre: "Estudio Chamberí"})
CREATE (a)-[:UBICADO_EN {distancia: 0.5}]->(b)
""",
"""
MATCH (b:Barrio {nombre: "Centro"}), (a:Alojamiento {nombre: "Loft Centro"})
CREATE (a)-[:UBICADO_EN {distancia: 0.1}]->(b)
"""
]

# Ejecutar cada sentencia por separado
for stmt in cypher_ubicado_en:
    run_query(stmt.strip(), verbose=False)

print("Relaciones UBICADO_EN creadas.")

Relaciones UBICADO_EN creadas.


### 6.2 Relación `CERCANO_A` (entidades físicamente próximas)

In [46]:
sentencias_cercano = [
    """
    MATCH (l:Local {nombre: "Cafetería Goya"}), (t:Terraza {nombre: "Terraza Sol"})
    CREATE (l)-[:CERCANO_A {distancia: 0.2}]->(t)
    """,
    """
    MATCH (a:Alojamiento {nombre: "Airbnb Retiro"}), (l:Local {nombre: "Librería Central"})
    CREATE (a)-[:CERCANO_A {distancia: 0.1}]->(l)
    """,
    """
    MATCH (a:Alojamiento {nombre: "Suite Salamanca"}), (l:Local {nombre: "Cafetería Goya"})
    CREATE (a)-[:CERCANO_A {distancia: 0.3}]->(l)
    """,
    """
    MATCH (a:Alojamiento {nombre: "Loft Centro"}), (t:Terraza {nombre: "Terraza Norte"})
    CREATE (a)-[:CERCANO_A {distancia: 0.15}]->(t)
    """,
    """
    MATCH (l:Local {nombre: "Restaurante El Patio"}), (t:Terraza {nombre: "Terraza Norte"})
    CREATE (l)-[:CERCANO_A {distancia: 0.05}]->(t)
    """
]

for stmt in sentencias_cercano:
    run_query(stmt.strip(), verbose=False)

print("Relaciones CERCANO_A creadas.")

Relaciones CERCANO_A creadas.


### 6.3 Relación `RELACIONADO_CON` (atributos comunes)

In [47]:
sentencias_relacionado = [
    """
    MATCH (l:Local {tipo_actividad: "Restauración"}), (t:Terraza {nombre: "Terraza Sol"})
    CREATE (l)-[:RELACIONADO_CON {motivo: "Ambiente de restauración"}]->(t)
    """,
    """
    MATCH (l1:Local {nombre: "Cafetería Goya"}), (l2:Local {nombre: "Restaurante El Patio"})
    CREATE (l1)-[:RELACIONADO_CON {motivo: "Misma categoría de restauración"}]->(l2)
    """,
    """
    MATCH (a1:Alojamiento {nombre: "Suite Salamanca"}), (a2:Alojamiento {nombre: "Apartamento Goya"})
    CREATE (a1)-[:RELACIONADO_CON {motivo: "Mismo barrio Salamanca"}]->(a2)
    """
]

for stmt in sentencias_relacionado:
    run_query(stmt.strip(), verbose=False)

print("Relaciones RELACIONADO_CON creadas.")

Relaciones RELACIONADO_CON creadas.


## 7. Verificación del grafo creado

In [25]:
# Contar relaciones por tipo
print("--- Relaciones por tipo ---")
for rel_type in ["UBICADO_EN", "CERCANO_A", "RELACIONADO_CON"]:
    count = run_query(f"MATCH ()-[r:{rel_type}]->() RETURN COUNT(r) AS total", verbose=False)
    print(f"  {rel_type:<20}: {count[0]['total']} relaciones")

--- Relaciones por tipo ---
  UBICADO_EN          : 12 relaciones
  CERCANO_A           : 5 relaciones
  RELACIONADO_CON     : 4 relaciones


In [26]:
# Ver todos los nodos y relaciones del grafo
print("--- Vista general del grafo ---")
resultados = run_query("""
    MATCH (n)-[r]->(m)
    RETURN 
        labels(n)[0] AS origen_tipo,
        n.nombre     AS origen,
        type(r)      AS relacion,
        labels(m)[0] AS destino_tipo,
        m.nombre     AS destino
    ORDER BY type(r), origen
""", verbose=False)

print(f"{'Origen':<30} {'Relación':<22} {'Destino':<25}")
print("-" * 78)
for r in resultados:
    origen  = f"({r['origen_tipo']}) {r['origen']}"
    destino = f"({r['destino_tipo']}) {r['destino']}"
    print(f"  {origen:<30} --{r['relacion']}--> {destino}")

--- Vista general del grafo ---
Origen                         Relación               Destino                  
------------------------------------------------------------------------------
  (Alojamiento) Airbnb Retiro    --CERCANO_A--> (Local) Librería Central
  (Local) Cafetería Goya         --CERCANO_A--> (Terraza) Terraza Sol
  (Alojamiento) Loft Centro      --CERCANO_A--> (Terraza) Terraza Norte
  (Local) Restaurante El Patio   --CERCANO_A--> (Terraza) Terraza Norte
  (Alojamiento) Suite Salamanca  --CERCANO_A--> (Local) Cafetería Goya
  (Local) Cafetería Goya         --RELACIONADO_CON--> (Terraza) Terraza Sol
  (Local) Cafetería Goya         --RELACIONADO_CON--> (Local) Restaurante El Patio
  (Local) Restaurante El Patio   --RELACIONADO_CON--> (Terraza) Terraza Sol
  (Alojamiento) Suite Salamanca  --RELACIONADO_CON--> (Alojamiento) Apartamento Goya
  (Alojamiento) Airbnb Retiro    --UBICADO_EN--> (Barrio) Retiro
  (Alojamiento) Apartamento Goya --UBICADO_EN--> (Barrio) Salamanc

## 8. Consultas sobre el grafo

### Consulta 2.1: Todos los locales y terrazas del barrio "Salamanca"

In [27]:
print("=== Consulta 2.1: Locales y Terrazas en Salamanca ===")

cypher_2_1 = """
MATCH (b:Barrio {nombre: "Salamanca"})<-[:UBICADO_EN]-(n)
WHERE n:Local OR n:Terraza
RETURN n.nombre AS nombre, labels(n)[0] AS tipo
ORDER BY tipo, nombre
"""

resultados_2_1 = run_query(cypher_2_1, verbose=False)
print(f"{'Nombre':<30} {'Tipo':<12}")
print("-" * 44)
for r in resultados_2_1:
    print(f"  {r['nombre']:<30} {r['tipo']:<12}")
print(f"\nTotal: {len(resultados_2_1)} entidades")

=== Consulta 2.1: Locales y Terrazas en Salamanca ===
Nombre                         Tipo        
--------------------------------------------
  Cafetería Goya                 Local       
  Farmacia Mayor                 Local       
  Terraza Sol                    Terraza     

Total: 3 entidades


### Consulta 2.2: Alojamientos cuyo precio supere los 100€

In [28]:
print("=== Consulta 2.2: Alojamientos con precio > 100€ ===")

cypher_2_2 = """
MATCH (a:Alojamiento)
WHERE a.precio > 100
RETURN a.nombre AS nombre, a.precio AS precio
ORDER BY a.precio DESC
"""

resultados_2_2 = run_query(cypher_2_2, verbose=False)
print(f"{'Nombre':<30} {'Precio (€)':>12}")
print("-" * 44)
for r in resultados_2_2:
    print(f"  {r['nombre']:<30} {r['precio']:>12}")
print(f"\nTotal: {len(resultados_2_2)} alojamientos")

=== Consulta 2.2: Alojamientos con precio > 100€ ===
Nombre                           Precio (€)
--------------------------------------------
  Suite Salamanca                         200
  Loft Centro                             150
  Airbnb Retiro                           120

Total: 3 alojamientos


### Consulta 2.3: Barrios donde los alojamientos no cuentan con dormitorios

In [30]:
print("=== Consulta 2.3: Barrios con alojamientos sin dormitorios ===")

cypher_2_3 = """
MATCH (a:Alojamiento)-[:UBICADO_EN]->(b:Barrio)
WHERE a.numero_habitaciones = 0
RETURN b.nombre AS barrio, COUNT(a) AS total_alojamientos
ORDER BY total_alojamientos DESC
"""

resultados_2_3 = run_query(cypher_2_3, verbose=False)
print(f"{'Barrio':<20} {'Alojamientos sin dormitorio':>24}")
print("-" * 46)
for r in resultados_2_3:
    print(f"  {r['barrio']:<20} {r['total_alojamientos']:>24}")
print(f"\nTotal de barrios afectados: {len(resultados_2_3)}")

=== Consulta 2.3: Barrios con alojamientos sin dormitorios ===
Barrio               Alojamientos sin dormitorio
----------------------------------------------
  Chamberí                                    1
  Centro                                      1

Total de barrios afectados: 2


### Consulta 2.4: Barrios donde los alojamientos no cuentan con reseñas

In [31]:
print("=== Consulta 2.4: Barrios con alojamientos sin reseñas ===")

cypher_2_4 = """
MATCH (a:Alojamiento)-[:UBICADO_EN]->(b:Barrio)
WHERE a.reseñas = 0
RETURN b.nombre AS barrio, COUNT(a) AS total_alojamientos
ORDER BY total_alojamientos DESC
"""

resultados_2_4 = run_query(cypher_2_4, verbose=False)
print(f"{'Barrio':<20} {'Alojamientos sin reseñas':>26}")
print("-" * 48)
for r in resultados_2_4:
    print(f"  {r['barrio']:<20} {r['total_alojamientos']:>26}")
print(f"\nTotal de barrios afectados: {len(resultados_2_4)}")

=== Consulta 2.4: Barrios con alojamientos sin reseñas ===
Barrio                 Alojamientos sin reseñas
------------------------------------------------
  Salamanca                                     1
  Chamberí                                      1

Total de barrios afectados: 2


In [33]:
# Cerrar conexión
driver.close()
print("Conexión Neo4j cerrada.")

Conexión Neo4j cerrada.
